# Backpropagation on a Computation Graph

AI & ML Foundations: Neural networks and backpropagation · *Mathematics for AI and Machine Learning* by Fuheng Wu

Forward values and backward gradients node by node for a tiny network, checked against finite differences.

Companion code for the [interactive demo](https://math4ai.org/companion/ml/backprop-graph/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's defaults
x, y = 1.0, 0.5
w1, b1, w2, b2 = 0.8, 0.2, -1.2, 0.3

# ── Forward and backward by hand: one local derivative per edge ─────────────
z1 = w1 * x + b1
a1 = math.tanh(z1)
z2 = w2 * a1 + b2
L = 0.5 * (z2 - y) ** 2

dz2 = z2 - y                 # dL/dz2
da1 = dz2 * w2               # dz2/da1 = w2
dz1 = da1 * (1 - a1 ** 2)    # da1/dz1 = 1 - tanh^2
manual = {"w1": dz1 * x, "b1": dz1, "w2": dz2 * a1, "b2": dz2}
print(f"forward : z1 = {z1:.5f}, a1 = {a1:.5f}, z2 = {z2:.5f}, L = {L:.5f}")
print(f"backward: dL/dz2 = {dz2:.5f}, dL/da1 = {da1:.5f}, dL/dz1 = {dz1:.5f}")

# ── The same graph in PyTorch: autograd does exactly these steps ────────────
P = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in
     {"w1": w1, "b1": b1, "w2": w2, "b2": b2}.items()}
z1_t = P["w1"] * x + P["b1"]
a1_t = torch.tanh(z1_t)
z2_t = P["w2"] * a1_t + P["b2"]
for t in (z1_t, a1_t, z2_t):
    t.retain_grad()                         # keep gradients of intermediate nodes too
L_t = 0.5 * (z2_t - y) ** 2
L_t.backward()
print("autograd node grads: dL/dz2 = %.5f, dL/da1 = %.5f, dL/dz1 = %.5f" % (z2_t.grad, a1_t.grad, z1_t.grad))

# ── Check every parameter gradient with central finite differences ──────────
def loss(p):
    return 0.5 * (p["w2"] * math.tanh(p["w1"] * x + p["b1"]) + p["b2"] - y) ** 2


base = {"w1": w1, "b1": b1, "w2": w2, "b2": b2}
h = 1e-6
print("\nparam   manual      autograd    finite diff")
for k in base:
    up, dn = dict(base), dict(base)
    up[k] += h; dn[k] -= h
    fd = (loss(up) - loss(dn)) / (2 * h)
    print(f"{k:5s}  {manual[k]:+.6f}   {P[k].grad.item():+.6f}   {fd:+.6f}")

forward : z1 = 1.00000, a1 = 0.76159, z2 = -0.61391, L = 0.62040
backward: dL/dz2 = -1.11391, dL/da1 = 1.33670, dL/dz1 = 0.56138
autograd node grads: dL/dz2 = -1.11391, dL/da1 = 1.33670, dL/dz1 = 0.56138

param   manual      autograd    finite diff
w1     +0.561378   +0.561378   +0.561378
b1     +0.561378   +0.561378   +0.561378
w2     -0.848350   -0.848350   -0.848350
b2     -1.113913   -1.113913   -1.113913
